In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

IN_JSON = "data/lod3_texturas_meshes.json"     # LOD3 with facade photos (12_texturas)
JSON_FILE = "data/colegio_imperial_fotos_meshes.json"   # Photo version (the final model is 14_materiales)
BLEND_FILE = "blender/colegio_imperial_fotos.blend"
GLB_FILE = "blender/colegio_imperial_fotos.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"
ORIGIN = (440000.0, 4473700.0, 640.0)

# True orthophoto 2023 (10 cm) of the Ayuntamiento de Madrid, by WMS:
WMS = "https://servpub.madrid.es/georaster/ORTOFOTOS/ows"
LAYER = "ORTOFOTOS:ORTOFOTO_MADRID_2023"
ORTHO_DIR = "sources/ortofoto_2023"
ORTHO_FILE = ORTHO_DIR + "/manzana_10cm.jpg"
BBOX = (439958.0, 4473659.0, 440184.0, 4473885.0)   # The block plus about 20 m (EPSG:25830)
RESOLUTION = 0.1   # Meters per pixel
TILE = 500         # Pixels per tile side (50 m), to keep each WMS request small

ROOF_NORMAL_Z = 0.3   # A face is roof if its normal points up at least this much (0 = vertical, 1 = flat)

CLOUD = "data/nube_manzana_2023.laz"
TERRAIN_CELL = 2.0    # Grid size of the terrain, in meters
TERRAIN_WINDOW = 8    # Cells around a node where we look for the lowest ground (8 x 2 m = 16 m)
TERRAIN_STEP = 2.5    # Meters above that lowest ground that a node may be (streets slope about 4 %)
TERRAIN_MARGIN = 16   # Meters of terrain kept around the block (the streets)
GPKG = "data/catastro_manzana.gpkg"
TERRAIN_SINK = 0.05   # The terrain goes 5 cm down, so that it does not flicker against the bottom of the buildings

In [ ]:
# Celda 2
import os
import io
import json
import math
import subprocess
import requests
from PIL import Image

os.makedirs(ORTHO_DIR, exist_ok=True)

In [ ]:
# Celda 3
# Download the orthophoto tile by tile and join the tiles in one image (only if it is not on disk yet):
xmin, ymin, xmax, ymax = BBOX
width_px = int(round((xmax - xmin) / RESOLUTION))
height_px = int(round((ymax - ymin) / RESOLUTION))

if os.path.exists(ORTHO_FILE):
    print("Already downloaded:", ORTHO_FILE)
else:
    mosaic = Image.new("RGB", (width_px, height_px))
    for top in range(0, height_px, TILE):
        for left in range(0, width_px, TILE):
            w = min(TILE, width_px - left)
            h = min(TILE, height_px - top)
            # Tile corners in meters (image rows go from north to south):
            tx0 = xmin + left * RESOLUTION
            ty1 = ymax - top * RESOLUTION
            bbox = [tx0, ty1 - h * RESOLUTION, tx0 + w * RESOLUTION, ty1]
            params = {"service": "WMS", "version": "1.3.0", "request": "GetMap", "layers": LAYER, "styles": "",
                      "crs": "EPSG:25830", "bbox": ",".join(str(round(v, 2)) for v in bbox),
                      "width": w, "height": h, "format": "image/jpeg"}
            answer = requests.get(WMS, params=params, timeout=120, headers={"User-Agent": "Mozilla/5.0"})
            if not answer.headers.get("Content-Type", "").startswith("image"):
                raise RuntimeError("WMS error: " + answer.text[:300])
            mosaic.paste(Image.open(io.BytesIO(answer.content)), (left, top))
    mosaic.save(ORTHO_FILE, quality=92)
    print("Saved:", ORTHO_FILE, width_px, "x", height_px, "px,", round(os.path.getsize(ORTHO_FILE) / 1e6, 1), "MB")

In [ ]:
# Celda 4
# Upward part of the normal of a face (polygon given by its vertices): 1 = flat roof, 0 = vertical wall.
def normal_z(points):
    nx, ny, nz = 0.0, 0.0, 0.0
    for k in range(len(points)):
        x1, y1, z1 = points[k]
        x2, y2, z2 = points[(k + 1) % len(points)]
        # Newell's method: add the cross products of consecutive vertices
        nx += (y1 - y2) * (z1 + z2)
        ny += (z1 - z2) * (x1 + x2)
        nz += (x1 - x2) * (y1 + y2)
    length = math.sqrt(nx * nx + ny * ny + nz * nz)
    if length == 0:
        return 0.0
    return nz / length

# Texture coordinate of a point seen from above: (xmin, ymin) -> (0, 0), (xmax, ymax) -> (1, 1):
def ortho_uv(x, y):
    return ((x + ORIGIN[0] - xmin) / (xmax - xmin), (y + ORIGIN[1] - ymin) / (ymax - ymin))

In [ ]:
# Celda 5
# Split every LOD1 / LOD2 building into its roof (with the orthophoto) and the rest (walls and bottom):
def sub_mesh(mesh, faces):
    new_index = {}
    vertices = []
    new_faces = []
    for face in faces:
        new_face = []
        for i in face:
            if i not in new_index:
                new_index[i] = len(vertices)
                vertices.append(mesh["vertices"][i])
            new_face.append(new_index[i])
        new_faces.append(tuple(new_face))
    part = dict(mesh)
    part["vertices"] = vertices
    part["faces"] = new_faces
    return part

with open(IN_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]

ortho_path = os.path.abspath(ORTHO_FILE)
result = []
roofs = 0
for mesh in meshes:
    if mesh.get("lod") not in (1, 2):
        result.append(mesh)
        continue
    roof_faces = []
    other_faces = []
    for face in mesh["faces"]:
        points = [mesh["vertices"][i] for i in face]
        if normal_z(points) >= ROOF_NORMAL_Z:
            roof_faces.append(face)
        else:
            other_faces.append(face)
    if other_faces:
        result.append(sub_mesh(mesh, other_faces))
    if roof_faces:
        roof = sub_mesh(mesh, roof_faces)
        roof["name"] = mesh["name"] + "_cubierta"
        roof["uv"] = [ortho_uv(x, y) for x, y, z in roof["vertices"]]
        roof["image"] = ortho_path
        result.append(roof)
        roofs += 1
print(len(meshes), "meshes in |", roofs, "roofs with the orthophoto |", len(result), "meshes out")

In [ ]:
# Celda 6
# Terrain (streets and courtyards) with the orthophoto: a grid of TERRAIN_CELL meters over BBOX.
# Each node takes the median of the ground points (class 2) closest to it; the nodes without ground
# (under buildings) take the mean of their filled neighbors, repeated until none is left.
import numpy as np
import laspy

cloud = laspy.read(CLOUD)
is_ground = np.array(cloud.classification) == 2
gx = np.array(cloud.x)[is_ground]
gy = np.array(cloud.y)[is_ground]
gz = np.array(cloud.z)[is_ground]

n_cols = int(round((xmax - xmin) / TERRAIN_CELL)) + 1
n_rows = int(round((ymax - ymin) / TERRAIN_CELL)) + 1
cols = np.round((gx - xmin) / TERRAIN_CELL).astype(int)
rows = np.round((gy - ymin) / TERRAIN_CELL).astype(int)
inside = (cols >= 0) & (cols < n_cols) & (rows >= 0) & (rows < n_rows)
keys = rows[inside] * n_cols + cols[inside]
heights = gz[inside]
order = np.argsort(keys)
keys = keys[order]
heights = heights[order]
unique_keys, starts = np.unique(keys, return_index=True)
Z = np.full((n_rows, n_cols), np.nan)
for key, values in zip(unique_keys, np.split(heights, starts[1:])):
    Z[key // n_cols, key % n_cols] = np.percentile(values, 10)
print(int(np.isnan(Z).sum()), "of", Z.size, "nodes without ground points")

# Many flat roofs of the cloud are classified as ground. A node is real ground only if it is close to the
# lowest ground around it: (node - lowest node within TERRAIN_WINDOW cells) <= TERRAIN_STEP meters.
rejected = 0
low = Z.copy()
for r in range(n_rows):
    for c in range(n_cols):
        window = Z[max(r - TERRAIN_WINDOW, 0):r + TERRAIN_WINDOW + 1, max(c - TERRAIN_WINDOW, 0):c + TERRAIN_WINDOW + 1]
        if not np.isnan(Z[r, c]) and Z[r, c] - np.nanmin(window) > TERRAIN_STEP:
            low[r, c] = np.nan
            rejected += 1
Z = low
print(rejected, "nodes rejected as roofs |", int(np.isnan(Z).sum()), "nodes to fill")

while np.isnan(Z).any():
    new_Z = Z.copy()
    for r in range(n_rows):
        for c in range(n_cols):
            if np.isnan(Z[r, c]):
                neighbors = Z[max(r - 1, 0):r + 2, max(c - 1, 0):c + 2]
                if not np.isnan(neighbors).all():
                    new_Z[r, c] = np.nanmean(neighbors)
    Z = new_Z

x0, y0, z0 = ORIGIN
vertices = []
uv = []
for r in range(n_rows):
    for c in range(n_cols):
        x = xmin + c * TERRAIN_CELL - x0
        y = ymin + r * TERRAIN_CELL - y0
        vertices.append((x, y, Z[r, c] - TERRAIN_SINK - z0))
        uv.append(ortho_uv(x, y))
# Keep only the cells near the block (the streets around it); farther away the cloud has no real ground:
import shapely
import geopandas as gpd
block = gpd.read_file(GPKG, layer="parcelas").geometry.union_all().buffer(TERRAIN_MARGIN)
faces = []
for r in range(n_rows - 1):
    for c in range(n_cols - 1):
        center_x = xmin + (c + 0.5) * TERRAIN_CELL
        center_y = ymin + (r + 0.5) * TERRAIN_CELL
        if block.contains(shapely.Point(center_x, center_y)):
            a = r * n_cols + c
            faces.append((a, a + 1, a + n_cols + 1, a + n_cols))
result.append({"name": "terreno", "parcel": "", "group": "contexto", "lod": 0, "height_m": 0.0,
               "height_method": "suelo de la nube 2023", "vertices": vertices, "faces": faces,
               "uv": uv, "image": ortho_path})
print("Terrain:", n_cols, "x", n_rows, "nodes |", len(result), "meshes in total")

In [ ]:
# Celda 7
# Save and run Blender:
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": result}, f)

command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
run = subprocess.run(command, capture_output=True, text=True)
for line_text in run.stdout.splitlines():
    if "Objects" in line_text or "Error" in line_text or "Saved" in line_text:
        print(line_text)
print("Return code:", run.returncode)
if run.returncode != 0:
    print(run.stderr[-2000:])
for path in [BLEND_FILE, GLB_FILE]:
    print(path, round(os.path.getsize(path) / 1e6, 2), "MB")